# TD Progressif : Valorisation des Produits Financiers avec Optionalité

**Objectif pédagogique** : Ce TD vous guidera progressivement des concepts fondamentaux de valorisation des instruments financiers jusqu'aux méthodes Monte Carlo pour les produits dérivés complexes.

---

## Table des matières

1. **TD1** - Obligations et Zéro-Coupons
2. **TD2** - Absence d'Opportunité d'Arbitrage (AOA) et Change à Terme
3. **TD3** - Valorisation d'un Call : Modèle Binomial à 1 Période
4. **TD4** - Extension : Arbre Binomial Multi-Périodes
5. **TD5** - Introduction aux Méthodes Monte Carlo
6. **TD6** - Options Exotiques et Réduction de Variance

---

In [ ]:
# Imports nécessaires pour l'ensemble du TD
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import brentq
import warnings
warnings.filterwarnings('ignore')

# Configuration des graphiques
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("✓ Environnement initialisé avec succès")

---

# TD1 : Obligations et Zéro-Coupons

## 1.1 Rappels théoriques

Une **obligation zéro-coupon** verse un flux fixe $N$ (le nominal) à une date future $T$. Son prix en $t$ est :

$$P(t,T) = N \times B(t,T)$$

où $B(t,T)$ est le **prix zéro-coupon** (discount factor). Le **taux zéro-coupon** $R(t,T)$ est défini par :

$$B(t,T) = \frac{1}{(1 + R(t,T))^{(T-t)}}$$

Une **OAT à taux fixe** de coupon $C = S \times N$ se valorise comme une combinaison de zéro-coupons :

$$P_\tau = \sum_{T_i > \tau} C \cdot B(\tau, T_i) + N \cdot B(\tau, T_n)$$

## Exercice 1.1 : Calcul du prix d'un zéro-coupon

**Énoncé** : Calculez le prix d'un zéro-coupon de nominal 100€ à différentes maturités pour un taux $R = 3\%$.

In [ ]:
def prix_zero_coupon(nominal: float, taux: float, maturite: float) -> float:
    """
    Calcule le prix d'un zéro-coupon.
    
    Parameters
    ----------
    nominal : float
        Valeur nominale (payée à maturité)
    taux : float
        Taux zéro-coupon annuel (ex: 0.03 pour 3%)
    maturite : float
        Durée jusqu'à l'échéance en années
    
    Returns
    -------
    float
        Prix actuel du zéro-coupon
    """
    B = 1 / (1 + taux) ** maturite
    return nominal * B

# Application
N = 100  # Nominal
R = 0.03  # Taux 3%
maturites = [1, 2, 3, 5, 10, 20, 30]

print("Prix d'un zéro-coupon de nominal 100€ au taux de 3%\n")
print(f"{'Maturité (ans)':<15} {'Prix (€)':<12} {'Discount B(0,T)':<15}")
print("-" * 42)

for T in maturites:
    prix = prix_zero_coupon(N, R, T)
    B = prix / N
    print(f"{T:<15} {prix:<12.4f} {B:<15.6f}")

## Exercice 1.2 : Valorisation d'une OAT à taux fixe

**Énoncé** : Une OAT de nominal 1000€, de taux facial 4%, verse des coupons annuels pendant 5 ans. La courbe des taux zéro-coupon est donnée ci-dessous. Calculez le prix de l'OAT.

In [ ]:
def prix_oat(nominal: float, taux_facial: float, 
             maturites_coupons: list, taux_zc: list) -> dict:
    """
    Valorise une OAT à taux fixe.
    
    Parameters
    ----------
    nominal : float
        Valeur nominale de l'obligation
    taux_facial : float
        Taux du coupon annuel
    maturites_coupons : list
        Dates de versement des coupons (en années)
    taux_zc : list
        Taux zéro-coupon correspondants
    
    Returns
    -------
    dict
        Détail de la valorisation
    """
    coupon = taux_facial * nominal
    details = []
    prix_total = 0
    
    for i, (T, R) in enumerate(zip(maturites_coupons, taux_zc)):
        B = 1 / (1 + R) ** T
        
        # Dernier flux = coupon + nominal
        if i == len(maturites_coupons) - 1:
            flux = coupon + nominal
            nature = "Coupon + Nominal"
        else:
            flux = coupon
            nature = "Coupon"
        
        va = flux * B
        prix_total += va
        details.append({
            'Année': T,
            'Nature': nature,
            'Flux': flux,
            'Taux ZC': f"{R:.2%}",
            'B(0,T)': B,
            'VA Flux': va
        })
    
    return {'prix': prix_total, 'details': pd.DataFrame(details)}

# Données du problème
nominal = 1000
taux_facial = 0.04
maturites = [1, 2, 3, 4, 5]
# Courbe des taux (légèrement croissante)
taux_zc = [0.025, 0.028, 0.030, 0.032, 0.035]

# Calcul
resultat = prix_oat(nominal, taux_facial, maturites, taux_zc)

print("Valorisation de l'OAT\n")
print(f"Nominal: {nominal}€ | Taux facial: {taux_facial:.1%} | Coupon annuel: {taux_facial*nominal}€\n")
print(resultat['details'].to_string(index=False))
print(f"\n{'='*60}")
print(f"PRIX DE L'OAT : {resultat['prix']:.2f}€")
print(f"Prime/Décote par rapport au pair : {resultat['prix'] - nominal:+.2f}€")

## Exercice 1.3 : Stripping de la courbe des taux

**Énoncé** : À partir des prix observés de 3 OAT, retrouvez les taux zéro-coupon par la méthode du bootstrap.

In [ ]:
def bootstrap_courbe_taux(obligations: list) -> pd.DataFrame:
    """
    Extrait la courbe des taux zéro-coupon par bootstrap.
    
    Parameters
    ----------
    obligations : list of dict
        Liste des obligations avec 'prix', 'coupon', 'maturite', 'nominal'
    
    Returns
    -------
    pd.DataFrame
        Courbe des taux zéro-coupon
    """
    # Tri par maturité
    obligations = sorted(obligations, key=lambda x: x['maturite'])
    
    taux_zc = {}  # T -> R(0,T)
    resultats = []
    
    for oblig in obligations:
        T = oblig['maturite']
        P = oblig['prix']
        C = oblig['coupon']
        N = oblig['nominal']
        
        # Valeur actualisée des coupons déjà connus
        va_coupons_connus = 0
        for t in range(1, T):
            if t in taux_zc:
                B_t = 1 / (1 + taux_zc[t]) ** t
                va_coupons_connus += C * B_t
        
        # Résolution : P = VA_coupons_connus + (C + N) * B(0,T)
        # B(0,T) = (P - VA_coupons_connus) / (C + N)
        B_T = (P - va_coupons_connus) / (C + N)
        R_T = B_T ** (-1/T) - 1
        
        taux_zc[T] = R_T
        resultats.append({
            'Maturité': T,
            'Prix OAT': P,
            'B(0,T)': B_T,
            'Taux ZC': R_T
        })
    
    return pd.DataFrame(resultats)

# Données de marché (OAT cotées)
obligations = [
    {'prix': 101.50, 'coupon': 4.0, 'maturite': 1, 'nominal': 100},
    {'prix': 103.20, 'coupon': 4.5, 'maturite': 2, 'nominal': 100},
    {'prix': 104.80, 'coupon': 5.0, 'maturite': 3, 'nominal': 100},
]

courbe = bootstrap_courbe_taux(obligations)
print("Stripping de la courbe des taux par bootstrap\n")
print(courbe.to_string(index=False, formatters={
    'B(0,T)': '{:.6f}'.format,
    'Taux ZC': '{:.4%}'.format
}))

# Visualisation
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(courbe['Maturité'], courbe['Taux ZC'] * 100, 'bo-', markersize=10, linewidth=2)
ax.set_xlabel('Maturité (années)')
ax.set_ylabel('Taux zéro-coupon (%)')
ax.set_title('Courbe des Taux Zéro-Coupon (Bootstrap)')
ax.set_xticks(courbe['Maturité'])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

# TD2 : Absence d'Opportunité d'Arbitrage (AOA) et Change à Terme

## 2.1 Rappels théoriques

**Définition (AOA)** : Une opportunité d'arbitrage est une stratégie à investissement nul qui rapporte des gains toujours positifs et strictement positifs avec probabilité non nulle.

**Taux de change à terme** : Pour garantir l'AOA, le taux de change forward $X(t,T)$ doit vérifier :

$$X(t,T) = X(t) \times \frac{B_{domestic}(t,T)}{B_{foreign}(t,T)}$$

## Exercice 2.1 : Détection d'opportunité d'arbitrage

**Énoncé** : Deux zéro-coupons de maturité 1 an et 2 ans ont les prix suivants. Y a-t-il une opportunité d'arbitrage ?

In [ ]:
def detecter_arbitrage_taux(B_1: float, B_2: float) -> dict:
    """
    Détecte une opportunité d'arbitrage entre deux zéro-coupons.
    
    L'arbitrage est possible si le taux forward implicite est négatif
    ou si les prix violent la cohérence temporelle.
    """
    # Taux ZC
    R_1 = (1/B_1) - 1
    R_2 = (1/B_2) ** 0.5 - 1
    
    # Taux forward 1 an dans 1 an (implicite)
    # B_2 = B_1 * B_forward => B_forward = B_2 / B_1
    B_fwd = B_2 / B_1
    R_fwd = 1/B_fwd - 1
    
    arbitrage = False
    strategie = None
    
    # Condition d'arbitrage : taux forward négatif
    if R_fwd < 0:
        arbitrage = True
        strategie = """Stratégie d'arbitrage :
        1. Emprunter 1€ à 2 ans (coût: {:.4f}€ en t=2)
        2. Placer 1€ à 1 an (gain: {:.4f}€ en t=1)
        3. En t=1, placer ce gain au taux spot 1 an
        Gain sans risque si le taux forward spot en t=1 > {:.2%}""".format(
            1/B_2, 1/B_1, R_fwd)
    
    return {
        'R_1': R_1,
        'R_2': R_2,
        'R_forward': R_fwd,
        'arbitrage': arbitrage,
        'strategie': strategie
    }

# Cas 1 : Marché cohérent
print("=" * 60)
print("CAS 1 : Marché cohérent")
print("=" * 60)
B_1 = 0.97  # Prix ZC 1 an
B_2 = 0.93  # Prix ZC 2 ans

result = detecter_arbitrage_taux(B_1, B_2)
print(f"B(0,1) = {B_1:.4f} => R(0,1) = {result['R_1']:.2%}")
print(f"B(0,2) = {B_2:.4f} => R(0,2) = {result['R_2']:.2%}")
print(f"Taux forward implicite f(1,2) = {result['R_forward']:.2%}")
print(f"\nArbitrage possible ? {'OUI ⚠️' if result['arbitrage'] else 'NON ✓'}")

# Cas 2 : Opportunité d'arbitrage
print("\n" + "=" * 60)
print("CAS 2 : Marché incohérent")
print("=" * 60)
B_1 = 0.95  # Prix ZC 1 an
B_2 = 0.96  # Prix ZC 2 ans (incohérent : devrait être < B_1)

result = detecter_arbitrage_taux(B_1, B_2)
print(f"B(0,1) = {B_1:.4f} => R(0,1) = {result['R_1']:.2%}")
print(f"B(0,2) = {B_2:.4f} => R(0,2) = {result['R_2']:.2%}")
print(f"Taux forward implicite f(1,2) = {result['R_forward']:.2%}")
print(f"\nArbitrage possible ? {'OUI ⚠️' if result['arbitrage'] else 'NON ✓'}")
if result['strategie']:
    print(f"\n{result['strategie']}")

## Exercice 2.2 : Taux de change à terme

**Énoncé** : Un industriel français doit payer 5M USD dans 6 mois. Calculez le taux EUR/USD forward et construisez la stratégie de couverture.

In [ ]:
def change_forward(spot: float, B_dom: float, B_for: float, 
                   montant_for: float) -> dict:
    """
    Calcule le taux de change forward et la stratégie de couverture.
    
    Parameters
    ----------
    spot : float
        Taux de change spot (domestique/étranger, ex: EUR/USD)
    B_dom : float
        Prix zéro-coupon domestique
    B_for : float
        Prix zéro-coupon étranger
    montant_for : float
        Montant en devise étrangère à couvrir
    """
    # Formule de parité couverte des taux d'intérêt
    forward = spot * B_dom / B_for
    
    # Stratégie de réplication pour un achat à terme de devises étrangères
    # 1. Acheter aujourd'hui un ZC étranger de nominal = montant_for
    cout_zc_etranger = montant_for * B_for  # en devise étrangère
    cout_en_dom_spot = cout_zc_etranger * spot  # converti au spot
    
    # 2. Financer par emprunt domestique
    remboursement_dom = cout_en_dom_spot / B_dom
    
    return {
        'forward': forward,
        'cout_replication': remboursement_dom,
        'strategie': {
            'achat_zc_etranger': cout_zc_etranger,
            'cout_spot': cout_en_dom_spot,
            'emprunt_dom': cout_en_dom_spot,
            'remboursement': remboursement_dom
        }
    }

# Données de marché
spot_eurusd = 1.08  # 1 EUR = 1.08 USD => pour acheter 1 USD il faut 1/1.08 EUR
spot_usdeur = 1 / spot_eurusd  # 1 USD = 0.926 EUR
B_eur_6m = 0.98  # ZC EUR 6 mois
B_usd_6m = 0.975  # ZC USD 6 mois
montant_usd = 5_000_000

result = change_forward(spot_usdeur, B_eur_6m, B_usd_6m, montant_usd)

print("Couverture de change à terme\n")
print(f"Spot EUR/USD : {spot_eurusd:.4f} (1 EUR = {spot_eurusd:.4f} USD)")
print(f"Spot USD/EUR : {spot_usdeur:.4f} (1 USD = {spot_usdeur:.4f} EUR)")
print(f"ZC EUR 6M : {B_eur_6m:.4f} => Taux EUR : {(1/B_eur_6m - 1)*2:.2%} (annualisé)")
print(f"ZC USD 6M : {B_usd_6m:.4f} => Taux USD : {(1/B_usd_6m - 1)*2:.2%} (annualisé)")
print(f"\nMontant à couvrir : {montant_usd:,.0f} USD")
print(f"\n{'='*50}")
print(f"Taux forward USD/EUR (6M) : {result['forward']:.6f}")
print(f"Taux forward EUR/USD (6M) : {1/result['forward']:.4f}")
print(f"\nCoût total en EUR dans 6 mois : {result['cout_replication']:,.2f} EUR")
print(f"{'='*50}")

print("\n📋 Stratégie de couverture (point de vue banque) :")
print(f"   1. Acheter aujourd'hui un ZC USD de nominal {montant_usd:,.0f} USD")
print(f"      Coût : {result['strategie']['achat_zc_etranger']:,.2f} USD")
print(f"   2. Convertir au spot : {result['strategie']['cout_spot']:,.2f} EUR")
print(f"   3. Emprunter {result['strategie']['emprunt_dom']:,.2f} EUR à 6M")
print(f"   4. En T : recevoir {montant_usd:,.0f} USD, rembourser {result['strategie']['remboursement']:,.2f} EUR")

## Exercice 2.3 : Prix forward d'un actif et opération repo

**Énoncé** : Une action cote 150€. Le taux repo à 3 mois est de 2% (annualisé). Calculez le prix forward.

In [ ]:
def prix_forward_actif(spot: float, taux_repo: float, maturite: float, 
                       dividendes: list = None) -> dict:
    """
    Calcule le prix forward d'un actif avec ou sans dividendes.
    
    Parameters
    ----------
    spot : float
        Prix spot de l'actif
    taux_repo : float
        Taux repo annualisé
    maturite : float
        Maturité du forward en années
    dividendes : list of tuples, optional
        Liste de (date, montant) des dividendes
    """
    # Cas sans dividende : F = S * (1 + r_repo)^T
    forward_sans_div = spot * (1 + taux_repo) ** maturite
    
    # Avec dividendes : on soustrait la VA des dividendes du spot
    va_dividendes = 0
    if dividendes:
        for date_div, montant in dividendes:
            if date_div < maturite:
                va_dividendes += montant * (1 + taux_repo) ** (maturite - date_div)
    
    forward_avec_div = forward_sans_div - va_dividendes
    
    return {
        'forward_sans_div': forward_sans_div,
        'forward_avec_div': forward_avec_div,
        'va_dividendes_en_T': va_dividendes,
        'cost_of_carry': (forward_sans_div / spot - 1) / maturite
    }

# Sans dividende
S0 = 150
r_repo = 0.02
T = 0.25  # 3 mois

result = prix_forward_actif(S0, r_repo, T)
print("Prix Forward d'un Actif\n")
print(f"Spot : {S0}€")
print(f"Taux repo : {r_repo:.2%}")
print(f"Maturité : {T*12:.0f} mois")
print(f"\nPrix forward (sans dividende) : {result['forward_sans_div']:.2f}€")
print(f"Cost of carry annualisé : {result['cost_of_carry']:.2%}")

# Avec dividende
print("\n" + "-"*50)
print("Cas avec un dividende de 3€ versé dans 1 mois\n")

dividendes = [(1/12, 3)]  # Dividende de 3€ dans 1 mois
result_div = prix_forward_actif(S0, r_repo, T, dividendes)

print(f"VA du dividende capitalisée en T : {result_div['va_dividendes_en_T']:.2f}€")
print(f"Prix forward (avec dividende) : {result_div['forward_avec_div']:.2f}€")

---

# TD3 : Valorisation d'un Call - Modèle Binomial à 1 Période

## 3.1 Rappels théoriques

Dans un univers à 2 dates ($t$ et $T$) et 2 états du monde (haut/bas), sous AOA :

$$C = \frac{\Pi_h \cdot C_h + \Pi_b \cdot C_b}{1 + r}$$

avec les **probabilités risque-neutre** :

$$\Pi_h = \frac{S(1+r) - S_b}{S_h - S_b}, \quad \Pi_b = 1 - \Pi_h$$

**Équivalence fondamentale** : AOA $\Leftrightarrow$ Existence d'une probabilité risque-neutre $\mathbb{Q}$

## Exercice 3.1 : Valorisation par réplication

**Énoncé** : Une action cote 100€. Dans 1 an, elle vaudra soit 130€ (hausse) soit 80€ (baisse). Le taux sans risque est 5%. Valorisez un Call de strike 110€.

In [ ]:
class ModèleBinomial1P:
    """
    Modèle binomial à une période pour la valorisation d'options.
    """
    
    def __init__(self, S0: float, Sh: float, Sb: float, r: float, T: float = 1):
        """
        Parameters
        ----------
        S0 : float - Prix spot
        Sh : float - Prix dans l'état haut
        Sb : float - Prix dans l'état bas
        r : float - Taux sans risque
        T : float - Maturité
        """
        self.S0 = S0
        self.Sh = Sh
        self.Sb = Sb
        self.r = r
        self.T = T
        
        # Vérification AOA
        if not (Sb < S0 * (1 + r) ** T < Sh):
            raise ValueError("Violation de l'AOA: Sb < S0(1+r)^T < Sh non respecté")
        
        # Probabilités risque-neutre
        self.q = (S0 * (1 + r) ** T - Sb) / (Sh - Sb)
        self.p = 1 - self.q
    
    def valoriser_option(self, K: float, option_type: str = 'call') -> dict:
        """
        Valorise une option européenne.
        
        Returns
        -------
        dict avec prix, delta, et portefeuille répliquant
        """
        # Payoffs
        if option_type.lower() == 'call':
            Ch = max(self.Sh - K, 0)
            Cb = max(self.Sb - K, 0)
        else:  # put
            Ch = max(K - self.Sh, 0)
            Cb = max(K - self.Sb, 0)
        
        # Portefeuille répliquant : α en actif sans risque, β en action
        # β * Sh + α * (1+r) = Ch
        # β * Sb + α * (1+r) = Cb
        beta = (Ch - Cb) / (self.Sh - self.Sb)  # Delta
        alpha = (Ch - beta * self.Sh) / ((1 + self.r) ** self.T)
        
        # Prix par réplication
        prix_replication = alpha + beta * self.S0
        
        # Prix par probabilité risque-neutre
        prix_rn = (self.q * Ch + self.p * Cb) / ((1 + self.r) ** self.T)
        
        return {
            'prix': prix_replication,
            'prix_rn': prix_rn,
            'delta': beta,
            'cash': alpha,
            'payoff_h': Ch,
            'payoff_b': Cb,
            'prob_rn_h': self.q,
            'prob_rn_b': self.p
        }
    
    def afficher_arbre(self, K: float, option_type: str = 'call'):
        """Affiche l'arbre binomial."""
        res = self.valoriser_option(K, option_type)
        
        print(f"\n{'='*60}")
        print(f"ARBRE BINOMIAL - {option_type.upper()} Strike {K}")
        print(f"{'='*60}\n")
        
        print(f"                           t=0                    t=T")
        print(f"                            │")
        print(f"                            │     ┌─── État Haut ───┐")
        print(f"                            │     │  S = {self.Sh:.2f}")
        print(f"           S = {self.S0:.2f}  ─────┤     │  Payoff = {res['payoff_h']:.2f}")
        print(f"           C = {res['prix']:.2f}         │  q = {self.q:.4f}")
        print(f"                            │     │")
        print(f"                            │     └─── État Bas ────┐")
        print(f"                            │        S = {self.Sb:.2f}")
        print(f"                            └───────  Payoff = {res['payoff_b']:.2f}")
        print(f"                                   1-q = {self.p:.4f}")
        
        print(f"\n{'─'*60}")
        print(f"Portefeuille répliquant :")
        print(f"  • Δ (delta) = {res['delta']:.4f} actions")
        print(f"  • Cash = {res['cash']:.4f}€ (emprunt si < 0)")
        print(f"\nVérification : {res['delta']:.4f} × {self.S0} + {res['cash']:.4f} = {res['prix']:.4f}€")


# Application
modele = ModèleBinomial1P(S0=100, Sh=130, Sb=80, r=0.05)

# Call
result_call = modele.valoriser_option(K=110, option_type='call')
modele.afficher_arbre(K=110, option_type='call')

print(f"\n\n{'='*60}")
print("RÉSULTATS")
print(f"{'='*60}")
print(f"Prix du Call (par réplication) : {result_call['prix']:.4f}€")
print(f"Prix du Call (par proba RN)    : {result_call['prix_rn']:.4f}€")
print(f"\nProbabilités risque-neutre : q = {result_call['prob_rn_h']:.4f}, 1-q = {result_call['prob_rn_b']:.4f}")

## Exercice 3.2 : Put et parité Call-Put

**Énoncé** : Avec les mêmes données, valorisez le Put de strike 110€ et vérifiez la parité Call-Put.

In [ ]:
# Put
result_put = modele.valoriser_option(K=110, option_type='put')
modele.afficher_arbre(K=110, option_type='put')

# Vérification parité Call-Put
K = 110
S0 = 100
r = 0.05
T = 1

# Parité : C - P = S - K * B(0,T)
call_moins_put = result_call['prix'] - result_put['prix']
s_moins_k_actualise = S0 - K / (1 + r) ** T

print(f"\n{'='*60}")
print("VÉRIFICATION PARITÉ CALL-PUT")
print(f"{'='*60}")
print(f"\nFormule : C - P = S₀ - K × B(0,T)")
print(f"\nC - P = {result_call['prix']:.4f} - {result_put['prix']:.4f} = {call_moins_put:.4f}")
print(f"S₀ - K×B = {S0} - {K} × {1/(1+r):.4f} = {s_moins_k_actualise:.4f}")
print(f"\n✓ Parité vérifiée !" if abs(call_moins_put - s_moins_k_actualise) < 1e-10 else "✗ Erreur !")

## Exercice 3.3 : Marché incomplet - Pricing par sur-réplication

**Énoncé** : On considère maintenant 3 états du monde (haut, moyen, bas). Le marché est incomplet. Trouvez le prix de sur-réplication d'un actif X.

In [ ]:
from scipy.optimize import linprog

def pricing_sur_replication(S0: float, S_etats: list, X_payoffs: list, r: float = 0) -> dict:
    """
    Pricing par sur-réplication dans un marché incomplet.
    
    Le vendeur cherche le prix minimal tel qu'il puisse
    se couvrir dans tous les états du monde.
    
    Parameters
    ----------
    S0 : float - Prix spot de l'actif de couverture
    S_etats : list - Prix de S dans chaque état
    X_payoffs : list - Payoffs de l'actif X à répliquer
    r : float - Taux sans risque
    """
    n_etats = len(S_etats)
    
    # Variables : [alpha, beta] où
    # alpha = position en actif sans risque (normalise à 1 en t=0)
    # beta = position en actif S
    
    # Objectif : minimiser le coût initial = alpha + beta * S0
    c = [1, S0]
    
    # Contraintes : dans chaque état, le portefeuille doit couvrir le payoff
    # alpha * (1+r) + beta * S_i >= X_i pour tout i
    # Soit : -alpha * (1+r) - beta * S_i <= -X_i
    
    A_ub = []
    b_ub = []
    for i in range(n_etats):
        A_ub.append([-(1 + r), -S_etats[i]])
        b_ub.append(-X_payoffs[i])
    
    # Résolution
    result = linprog(c, A_ub=A_ub, b_ub=b_ub, method='highs')
    
    if result.success:
        alpha, beta = result.x
        prix = alpha + beta * S0
        
        # Calcul des excédents de couverture
        excedents = []
        for i in range(n_etats):
            couverture = alpha * (1 + r) + beta * S_etats[i]
            excedents.append(couverture - X_payoffs[i])
        
        return {
            'prix': prix,
            'alpha': alpha,
            'beta': beta,
            'excedents': excedents,
            'success': True
        }
    else:
        return {'success': False, 'message': result.message}


# Données de l'exercice (voir annexe)
S0 = 3
S_etats = [4, 2, 1]  # Sh, Sm, Sb
X_payoffs = [2, 1, 1]  # Xh, Xm, Xb
r = 0

result = pricing_sur_replication(S0, S_etats, X_payoffs, r)

print("MARCHÉ INCOMPLET : PRICING PAR SUR-RÉPLICATION\n")
print(f"Actif S : S₀ = {S0}")
print(f"États : Sₕ = {S_etats[0]}, Sₘ = {S_etats[1]}, Sᵦ = {S_etats[2]}")
print(f"\nActif X à répliquer :")
print(f"Payoffs : Xₕ = {X_payoffs[0]}, Xₘ = {X_payoffs[1]}, Xᵦ = {X_payoffs[2]}")

if result['success']:
    print(f"\n{'='*50}")
    print(f"SOLUTION")
    print(f"{'='*50}")
    print(f"\nPrix de sur-réplication : {result['prix']:.4f}")
    print(f"\nPortefeuille de couverture :")
    print(f"  α (cash) = {result['alpha']:.4f}")
    print(f"  β (actif S) = {result['beta']:.4f}")
    
    print(f"\nVérification de la couverture :")
    etats_noms = ['Haut', 'Moyen', 'Bas']
    for i, (nom, Si, Xi, exc) in enumerate(zip(etats_noms, S_etats, X_payoffs, result['excedents'])):
        couv = result['alpha'] * (1 + r) + result['beta'] * Si
        print(f"  État {nom}: α×(1+r) + β×S = {couv:.4f} >= {Xi} (excédent: {exc:.4f})")

---

# TD4 : Arbre Binomial Multi-Périodes (Cox-Ross-Rubinstein)

## 4.1 Rappels théoriques

On étend le modèle à $N$ périodes. À chaque pas, le cours peut monter d'un facteur $u$ ou descendre d'un facteur $d$.

Pour faire converger vers Black-Scholes :
$$u = e^{\sigma\sqrt{\Delta t}}, \quad d = e^{-\sigma\sqrt{\Delta t}}, \quad \Delta t = T/N$$

Probabilité risque-neutre : $q = \frac{e^{r\Delta t} - d}{u - d}$

In [ ]:
class ArbreBinomialCRR:
    """
    Modèle de Cox-Ross-Rubinstein pour la valorisation d'options.
    """
    
    def __init__(self, S0: float, K: float, T: float, r: float, 
                 sigma: float, N: int, option_type: str = 'call'):
        self.S0 = S0
        self.K = K
        self.T = T
        self.r = r
        self.sigma = sigma
        self.N = N
        self.option_type = option_type.lower()
        
        # Paramètres CRR
        self.dt = T / N
        self.u = np.exp(sigma * np.sqrt(self.dt))
        self.d = 1 / self.u
        self.q = (np.exp(r * self.dt) - self.d) / (self.u - self.d)
        self.discount = np.exp(-r * self.dt)
    
    def prix_europeen(self) -> float:
        """Valorisation d'une option européenne par récurrence arrière."""
        # Arbre des prix
        prix = np.zeros((self.N + 1, self.N + 1))
        
        # Prix à maturité
        for j in range(self.N + 1):
            ST = self.S0 * (self.u ** (self.N - j)) * (self.d ** j)
            if self.option_type == 'call':
                prix[self.N, j] = max(ST - self.K, 0)
            else:
                prix[self.N, j] = max(self.K - ST, 0)
        
        # Récurrence arrière
        for i in range(self.N - 1, -1, -1):
            for j in range(i + 1):
                prix[i, j] = self.discount * (self.q * prix[i+1, j] + (1 - self.q) * prix[i+1, j+1])
        
        return prix[0, 0]
    
    def prix_americain(self) -> float:
        """Valorisation d'une option américaine (exercice anticipé possible)."""
        prix = np.zeros((self.N + 1, self.N + 1))
        
        # Prix à maturité
        for j in range(self.N + 1):
            ST = self.S0 * (self.u ** (self.N - j)) * (self.d ** j)
            if self.option_type == 'call':
                prix[self.N, j] = max(ST - self.K, 0)
            else:
                prix[self.N, j] = max(self.K - ST, 0)
        
        # Récurrence avec possibilité d'exercice anticipé
        for i in range(self.N - 1, -1, -1):
            for j in range(i + 1):
                Si = self.S0 * (self.u ** (i - j)) * (self.d ** j)
                continuation = self.discount * (self.q * prix[i+1, j] + (1 - self.q) * prix[i+1, j+1])
                
                if self.option_type == 'call':
                    exercice = max(Si - self.K, 0)
                else:
                    exercice = max(self.K - Si, 0)
                
                prix[i, j] = max(continuation, exercice)
        
        return prix[0, 0]
    
    def calculer_greeks(self, epsilon: float = 0.01) -> dict:
        """Calcul des Greeks par différences finies."""
        prix_base = self.prix_europeen()
        
        # Delta
        self_up = ArbreBinomialCRR(self.S0 * (1 + epsilon), self.K, self.T, 
                                    self.r, self.sigma, self.N, self.option_type)
        self_down = ArbreBinomialCRR(self.S0 * (1 - epsilon), self.K, self.T, 
                                      self.r, self.sigma, self.N, self.option_type)
        delta = (self_up.prix_europeen() - self_down.prix_europeen()) / (2 * epsilon * self.S0)
        
        # Gamma
        gamma = (self_up.prix_europeen() - 2 * prix_base + self_down.prix_europeen()) / (epsilon * self.S0) ** 2
        
        # Vega
        self_vega_up = ArbreBinomialCRR(self.S0, self.K, self.T, self.r, 
                                         self.sigma + epsilon, self.N, self.option_type)
        vega = (self_vega_up.prix_europeen() - prix_base) / epsilon
        
        # Theta
        self_theta = ArbreBinomialCRR(self.S0, self.K, self.T - epsilon, self.r, 
                                       self.sigma, self.N, self.option_type)
        theta = (self_theta.prix_europeen() - prix_base) / epsilon
        
        return {'delta': delta, 'gamma': gamma, 'vega': vega, 'theta': theta}


# Application
params = {
    'S0': 100,
    'K': 100,
    'T': 1,
    'r': 0.05,
    'sigma': 0.20,
    'N': 100
}

print("MODÈLE BINOMIAL COX-ROSS-RUBINSTEIN\n")
print(f"Paramètres : S₀={params['S0']}, K={params['K']}, T={params['T']}")
print(f"             r={params['r']:.1%}, σ={params['sigma']:.1%}, N={params['N']} pas")

# Call européen et américain
crr_call = ArbreBinomialCRR(**params, option_type='call')
crr_put = ArbreBinomialCRR(**params, option_type='put')

print(f"\n{'='*50}")
print("RÉSULTATS")
print(f"{'='*50}")
print(f"\n{'Option':<20} {'Européenne':>12} {'Américaine':>12}")
print("-" * 44)
print(f"{'Call'::<20} {crr_call.prix_europeen():>12.4f} {crr_call.prix_americain():>12.4f}")
print(f"{'Put'::<20} {crr_put.prix_europeen():>12.4f} {crr_put.prix_americain():>12.4f}")

# Greeks
greeks = crr_call.calculer_greeks()
print(f"\n{'='*50}")
print("GREEKS (Call)")
print(f"{'='*50}")
print(f"Delta : {greeks['delta']:.4f}")
print(f"Gamma : {greeks['gamma']:.4f}")
print(f"Vega  : {greeks['vega']:.4f}")
print(f"Theta : {greeks['theta']:.4f}")

## Exercice 4.1 : Convergence vers Black-Scholes

**Énoncé** : Vérifiez la convergence du modèle CRR vers le prix Black-Scholes lorsque $N \to \infty$.

In [ ]:
def black_scholes(S0, K, T, r, sigma, option_type='call'):
    """Formule de Black-Scholes."""
    d1 = (np.log(S0/K) + (r + sigma**2/2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    if option_type == 'call':
        return S0 * norm.cdf(d1) - K * np.exp(-r*T) * norm.cdf(d2)
    else:
        return K * np.exp(-r*T) * norm.cdf(-d2) - S0 * norm.cdf(-d1)

# Paramètres
S0, K, T, r, sigma = 100, 100, 1, 0.05, 0.20

# Prix Black-Scholes
bs_price = black_scholes(S0, K, T, r, sigma, 'call')

# Convergence CRR
N_values = [5, 10, 20, 50, 100, 200, 500]
crr_prices = []
errors = []

for N in N_values:
    crr = ArbreBinomialCRR(S0, K, T, r, sigma, N, 'call')
    prix = crr.prix_europeen()
    crr_prices.append(prix)
    errors.append(abs(prix - bs_price))

# Affichage
print("CONVERGENCE CRR → BLACK-SCHOLES\n")
print(f"Prix Black-Scholes : {bs_price:.6f}\n")
print(f"{'N':>6} {'Prix CRR':>12} {'Erreur':>12} {'Erreur (%)':>12}")
print("-" * 44)
for N, prix, err in zip(N_values, crr_prices, errors):
    print(f"{N:>6} {prix:>12.6f} {err:>12.6f} {100*err/bs_price:>11.4f}%")

# Graphique
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(N_values, crr_prices, 'bo-', markersize=8, label='CRR')
ax1.axhline(bs_price, color='r', linestyle='--', label='Black-Scholes')
ax1.set_xlabel('Nombre de pas (N)')
ax1.set_ylabel('Prix du Call')
ax1.set_title('Convergence du modèle CRR')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.loglog(N_values, errors, 'go-', markersize=8)
ax2.set_xlabel('Nombre de pas (N)')
ax2.set_ylabel('Erreur absolue')
ax2.set_title('Erreur de convergence (échelle log-log)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

# TD5 : Introduction aux Méthodes Monte Carlo

## 5.1 Rappels théoriques

Le prix d'un dérivé est l'espérance sous $\mathbb{Q}$ des flux actualisés :

$$C_0 = e^{-rT} \mathbb{E}^{\mathbb{Q}}[\Phi(S_T)]$$

L'estimateur Monte Carlo est :

$$\hat{C}_N = e^{-rT} \frac{1}{N} \sum_{i=1}^{N} \Phi(S_T^{(i)})$$

L'erreur statistique converge en $O(1/\sqrt{N})$.

In [ ]:
class MonteCarloPricer:
    """
    Valorisation Monte Carlo d'options européennes.
    """
    
    def __init__(self, S0: float, K: float, T: float, r: float, sigma: float):
        self.S0 = S0
        self.K = K
        self.T = T
        self.r = r
        self.sigma = sigma
    
    def simuler_ST(self, N: int, seed: int = None) -> np.ndarray:
        """Simulation exacte de ST sous Black-Scholes."""
        if seed is not None:
            np.random.seed(seed)
        
        Z = np.random.standard_normal(N)
        ST = self.S0 * np.exp((self.r - 0.5 * self.sigma**2) * self.T + 
                               self.sigma * np.sqrt(self.T) * Z)
        return ST
    
    def prix_call(self, N: int, seed: int = None) -> dict:
        """Prix d'un call européen par Monte Carlo."""
        ST = self.simuler_ST(N, seed)
        payoffs = np.maximum(ST - self.K, 0)
        
        prix = np.exp(-self.r * self.T) * np.mean(payoffs)
        std_payoff = np.std(payoffs, ddof=1)
        ic_95 = 1.96 * std_payoff / np.sqrt(N) * np.exp(-self.r * self.T)
        
        return {
            'prix': prix,
            'std': std_payoff * np.exp(-self.r * self.T),
            'ic_95': ic_95,
            'ic_lower': prix - ic_95,
            'ic_upper': prix + ic_95
        }
    
    def prix_put(self, N: int, seed: int = None) -> dict:
        """Prix d'un put européen par Monte Carlo."""
        ST = self.simuler_ST(N, seed)
        payoffs = np.maximum(self.K - ST, 0)
        
        prix = np.exp(-self.r * self.T) * np.mean(payoffs)
        std_payoff = np.std(payoffs, ddof=1)
        ic_95 = 1.96 * std_payoff / np.sqrt(N) * np.exp(-self.r * self.T)
        
        return {
            'prix': prix,
            'std': std_payoff * np.exp(-self.r * self.T),
            'ic_95': ic_95,
            'ic_lower': prix - ic_95,
            'ic_upper': prix + ic_95
        }


# Application
mc = MonteCarloPricer(S0=100, K=100, T=1, r=0.05, sigma=0.20)

print("VALORISATION MONTE CARLO - CALL EUROPÉEN\n")
print(f"Paramètres : S₀=100, K=100, T=1, r=5%, σ=20%")

# Comparaison avec Black-Scholes
bs_price = black_scholes(100, 100, 1, 0.05, 0.20, 'call')
print(f"\nPrix Black-Scholes (référence) : {bs_price:.4f}\n")

# Différentes valeurs de N
N_values = [1000, 10000, 100000, 1000000]

print(f"{'N':>10} {'Prix MC':>12} {'IC 95%':>20} {'Erreur':>10}")
print("-" * 55)

for N in N_values:
    result = mc.prix_call(N, seed=42)
    ic_str = f"[{result['ic_lower']:.4f}, {result['ic_upper']:.4f}]"
    erreur = abs(result['prix'] - bs_price)
    print(f"{N:>10,} {result['prix']:>12.4f} {ic_str:>20} {erreur:>10.4f}")

## Exercice 5.1 : Simulation de trajectoires (Schéma d'Euler)

**Énoncé** : Implémentez le schéma d'Euler pour simuler des trajectoires du sous-jacent.

In [ ]:
def simuler_trajectoires_euler(S0: float, r: float, sigma: float, 
                                T: float, M: int, N: int, seed: int = None):
    """
    Simulation de trajectoires par le schéma d'Euler.
    
    Parameters
    ----------
    S0 : float - Prix initial
    r : float - Taux sans risque
    sigma : float - Volatilité
    T : float - Maturité
    M : int - Nombre de pas de temps
    N : int - Nombre de trajectoires
    
    Returns
    -------
    t : array - Grille temporelle
    S : array de shape (N, M+1) - Trajectoires
    """
    if seed is not None:
        np.random.seed(seed)
    
    dt = T / M
    t = np.linspace(0, T, M + 1)
    
    # Simulation sur le log (pour garantir positivité)
    X = np.zeros((N, M + 1))
    X[:, 0] = np.log(S0)
    
    for i in range(M):
        Z = np.random.standard_normal(N)
        X[:, i+1] = X[:, i] + (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
    
    S = np.exp(X)
    return t, S


# Simulation et visualisation
t, trajectoires = simuler_trajectoires_euler(
    S0=100, r=0.05, sigma=0.20, T=1, M=252, N=10, seed=42
)

fig, ax = plt.subplots(figsize=(12, 6))

for i in range(10):
    ax.plot(t, trajectoires[i], alpha=0.7, linewidth=1)

ax.axhline(100, color='black', linestyle='--', alpha=0.5, label='S₀')
ax.set_xlabel('Temps (années)')
ax.set_ylabel('Prix du sous-jacent')
ax.set_title('Simulation de 10 trajectoires (Schéma d\'Euler)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercice 5.2 : Étude de la convergence

**Énoncé** : Visualisez la convergence de l'estimateur Monte Carlo et l'évolution de l'intervalle de confiance.

In [ ]:
def etude_convergence_mc(S0, K, T, r, sigma, N_max, seed=42):
    """Étude de la convergence de l'estimateur MC."""
    np.random.seed(seed)
    
    # Simulation de tous les payoffs
    Z = np.random.standard_normal(N_max)
    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    payoffs = np.maximum(ST - K, 0)
    discount = np.exp(-r * T)
    
    # Calcul cumulatif
    N_values = np.arange(100, N_max + 1, 100)
    prix_cumul = np.cumsum(payoffs) / np.arange(1, N_max + 1)
    var_cumul = np.array([payoffs[:n].var(ddof=1) if n > 1 else 0 for n in N_values])
    
    prix = prix_cumul[N_values - 1] * discount
    ic_95 = 1.96 * np.sqrt(var_cumul / N_values) * discount
    
    return N_values, prix, ic_95


# Paramètres
S0, K, T, r, sigma = 100, 100, 1, 0.05, 0.20
bs_price = black_scholes(S0, K, T, r, sigma, 'call')

N_values, prix_mc, ic_95 = etude_convergence_mc(S0, K, T, r, sigma, N_max=50000)

# Visualisation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Convergence du prix
ax1.plot(N_values, prix_mc, 'b-', alpha=0.7, label='Prix MC')
ax1.fill_between(N_values, prix_mc - ic_95, prix_mc + ic_95, 
                  alpha=0.3, color='blue', label='IC 95%')
ax1.axhline(bs_price, color='red', linestyle='--', linewidth=2, label='Black-Scholes')
ax1.set_xlabel('Nombre de simulations (N)')
ax1.set_ylabel('Prix estimé')
ax1.set_title('Convergence de l\'estimateur Monte Carlo')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Largeur de l'IC
ax2.loglog(N_values, 2 * ic_95, 'g-', linewidth=2, label='Largeur IC 95%')
ax2.loglog(N_values, 2 * ic_95[0] * np.sqrt(N_values[0] / N_values), 'r--', 
           linewidth=1, label='Décroissance en 1/√N')
ax2.set_xlabel('Nombre de simulations (N)')
ax2.set_ylabel('Largeur de l\'IC')
ax2.set_title('Convergence de l\'erreur statistique')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

# TD6 : Options Exotiques et Réduction de Variance

## 6.1 Options path-dependent

Certaines options dépendent de la trajectoire du sous-jacent :
- **Asiatique** : payoff basé sur la moyenne des cours
- **Barrière** : activation/désactivation selon un seuil
- **Lookback** : payoff basé sur le max/min atteint

In [ ]:
class OptionsExotiques:
    """
    Valorisation Monte Carlo d'options exotiques.
    """
    
    def __init__(self, S0: float, K: float, T: float, r: float, sigma: float, M: int = 252):
        self.S0 = S0
        self.K = K
        self.T = T
        self.r = r
        self.sigma = sigma
        self.M = M  # Nombre de pas de temps
    
    def simuler_trajectoires(self, N: int, seed: int = None):
        """Simule N trajectoires avec M pas de temps."""
        if seed is not None:
            np.random.seed(seed)
        
        dt = self.T / self.M
        X = np.zeros((N, self.M + 1))
        X[:, 0] = np.log(self.S0)
        
        for i in range(self.M):
            Z = np.random.standard_normal(N)
            X[:, i+1] = X[:, i] + (self.r - 0.5 * self.sigma**2) * dt + self.sigma * np.sqrt(dt) * Z
        
        return np.exp(X)
    
    def call_asiatique_arithmetique(self, N: int, seed: int = None) -> dict:
        """Call asiatique sur moyenne arithmétique."""
        S = self.simuler_trajectoires(N, seed)
        moyennes = S.mean(axis=1)  # Moyenne sur chaque trajectoire
        payoffs = np.maximum(moyennes - self.K, 0)
        
        prix = np.exp(-self.r * self.T) * payoffs.mean()
        std = np.exp(-self.r * self.T) * payoffs.std(ddof=1)
        ic_95 = 1.96 * std / np.sqrt(N)
        
        return {'prix': prix, 'std': std, 'ic_95': ic_95}
    
    def call_barriere_knock_out(self, N: int, barriere: float, seed: int = None) -> dict:
        """Call Down-and-Out (désactivé si S < barrière)."""
        S = self.simuler_trajectoires(N, seed)
        ST = S[:, -1]
        
        # Vérifie si la barrière a été touchée
        barriere_touchee = (S.min(axis=1) <= barriere)
        
        payoffs = np.where(barriere_touchee, 0, np.maximum(ST - self.K, 0))
        
        prix = np.exp(-self.r * self.T) * payoffs.mean()
        std = np.exp(-self.r * self.T) * payoffs.std(ddof=1)
        ic_95 = 1.96 * std / np.sqrt(N)
        
        pct_knockout = barriere_touchee.mean() * 100
        
        return {'prix': prix, 'std': std, 'ic_95': ic_95, 'pct_knockout': pct_knockout}
    
    def call_lookback_floating(self, N: int, seed: int = None) -> dict:
        """Call Lookback à strike flottant (strike = min sur la période)."""
        S = self.simuler_trajectoires(N, seed)
        ST = S[:, -1]
        S_min = S.min(axis=1)
        
        payoffs = ST - S_min  # Toujours > 0
        
        prix = np.exp(-self.r * self.T) * payoffs.mean()
        std = np.exp(-self.r * self.T) * payoffs.std(ddof=1)
        ic_95 = 1.96 * std / np.sqrt(N)
        
        return {'prix': prix, 'std': std, 'ic_95': ic_95}


# Application
exo = OptionsExotiques(S0=100, K=100, T=1, r=0.05, sigma=0.20, M=252)
N = 100000

print("VALORISATION D'OPTIONS EXOTIQUES\n")
print(f"Paramètres : S₀=100, K=100, T=1, r=5%, σ=20%")
print(f"Simulations : N={N:,}, M=252 pas/an\n")

# Référence : Call vanille BS
bs_call = black_scholes(100, 100, 1, 0.05, 0.20, 'call')
print(f"{'='*60}")
print(f"Référence - Call Vanille (Black-Scholes) : {bs_call:.4f}")
print(f"{'='*60}\n")

# Call Asiatique
asia = exo.call_asiatique_arithmetique(N, seed=42)
print(f"Call Asiatique (moyenne arithmétique) :")
print(f"  Prix : {asia['prix']:.4f} ± {asia['ic_95']:.4f}")
print(f"  → Moins cher que le vanille (moyenne lisse la volatilité)\n")

# Call Barrière
barriere = 80
barr = exo.call_barriere_knock_out(N, barriere, seed=42)
print(f"Call Down-and-Out (barrière = {barriere}) :")
print(f"  Prix : {barr['prix']:.4f} ± {barr['ic_95']:.4f}")
print(f"  Trajectoires knock-out : {barr['pct_knockout']:.1f}%")
print(f"  → Moins cher car risque de désactivation\n")

# Call Lookback
look = exo.call_lookback_floating(N, seed=42)
print(f"Call Lookback (strike flottant = min) :")
print(f"  Prix : {look['prix']:.4f} ± {look['ic_95']:.4f}")
print(f"  → Plus cher car payoff toujours positif")

## Exercice 6.1 : Réduction de variance - Variables antithétiques

**Énoncé** : Implémentez la méthode des variables antithétiques et mesurez la réduction de variance.

In [ ]:
def mc_antithetique(S0, K, T, r, sigma, N, seed=42):
    """
    Monte Carlo avec variables antithétiques.
    
    Pour chaque tirage Z, on utilise aussi -Z.
    """
    np.random.seed(seed)
    
    Z = np.random.standard_normal(N // 2)
    
    # Trajectoires avec Z et -Z
    ST_plus = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    ST_moins = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * (-Z))
    
    # Payoffs
    payoff_plus = np.maximum(ST_plus - K, 0)
    payoff_moins = np.maximum(ST_moins - K, 0)
    
    # Estimateur antithétique
    payoff_anti = (payoff_plus + payoff_moins) / 2
    
    prix = np.exp(-r * T) * payoff_anti.mean()
    std = np.exp(-r * T) * payoff_anti.std(ddof=1)
    ic_95 = 1.96 * std / np.sqrt(N // 2)
    
    return {'prix': prix, 'std': std, 'ic_95': ic_95}


def mc_standard(S0, K, T, r, sigma, N, seed=42):
    """Monte Carlo standard (sans réduction de variance)."""
    np.random.seed(seed)
    
    Z = np.random.standard_normal(N)
    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    payoffs = np.maximum(ST - K, 0)
    
    prix = np.exp(-r * T) * payoffs.mean()
    std = np.exp(-r * T) * payoffs.std(ddof=1)
    ic_95 = 1.96 * std / np.sqrt(N)
    
    return {'prix': prix, 'std': std, 'ic_95': ic_95}


# Comparaison
S0, K, T, r, sigma = 100, 100, 1, 0.05, 0.20
N = 100000
bs_price = black_scholes(S0, K, T, r, sigma, 'call')

std_result = mc_standard(S0, K, T, r, sigma, N)
anti_result = mc_antithetique(S0, K, T, r, sigma, N)

print("RÉDUCTION DE VARIANCE : VARIABLES ANTITHÉTIQUES\n")
print(f"Prix Black-Scholes : {bs_price:.4f}\n")

print(f"{'Méthode':<25} {'Prix':>10} {'IC 95%':>12} {'Variance':>12}")
print("-" * 60)
print(f"{'MC Standard':<25} {std_result['prix']:>10.4f} {std_result['ic_95']:>12.4f} {std_result['std']**2:>12.4f}")
print(f"{'MC Antithétique':<25} {anti_result['prix']:>10.4f} {anti_result['ic_95']:>12.4f} {anti_result['std']**2:>12.4f}")

reduction = 1 - (anti_result['std']**2) / (std_result['std']**2)
print(f"\nRéduction de variance : {reduction:.1%}")
print(f"Réduction de l'IC : {1 - anti_result['ic_95']/std_result['ic_95']:.1%}")

## Exercice 6.2 : Variable de contrôle

**Énoncé** : Utilisez le cours final $S_T$ comme variable de contrôle pour améliorer l'estimation du prix du Call.

In [ ]:
def mc_control_variate(S0, K, T, r, sigma, N, seed=42):
    """
    Monte Carlo avec variable de contrôle.
    
    On utilise ST comme contrôle car E[ST] = S0 * exp(rT) est connu.
    """
    np.random.seed(seed)
    
    Z = np.random.standard_normal(N)
    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    payoffs = np.maximum(ST - K, 0)
    
    # Espérance théorique de ST sous Q
    E_ST = S0 * np.exp(r * T)
    
    # Coefficient optimal de contrôle
    cov = np.cov(payoffs, ST)[0, 1]
    var_ST = np.var(ST, ddof=1)
    beta_opt = cov / var_ST
    
    # Estimateur avec contrôle
    payoffs_corr = payoffs - beta_opt * (ST - E_ST)
    
    prix = np.exp(-r * T) * payoffs_corr.mean()
    std = np.exp(-r * T) * payoffs_corr.std(ddof=1)
    ic_95 = 1.96 * std / np.sqrt(N)
    
    return {
        'prix': prix, 
        'std': std, 
        'ic_95': ic_95,
        'beta_opt': beta_opt,
        'correlation': np.corrcoef(payoffs, ST)[0, 1]
    }


# Comparaison
ctrl_result = mc_control_variate(S0, K, T, r, sigma, N)

print("RÉDUCTION DE VARIANCE : VARIABLE DE CONTRÔLE\n")
print(f"Variable de contrôle : ST (espérance connue = S₀ × e^(rT))\n")

print(f"{'Méthode':<25} {'Prix':>10} {'IC 95%':>12} {'Variance':>12}")
print("-" * 60)
print(f"{'MC Standard':<25} {std_result['prix']:>10.4f} {std_result['ic_95']:>12.4f} {std_result['std']**2:>12.4f}")
print(f"{'MC Contrôle':<25} {ctrl_result['prix']:>10.4f} {ctrl_result['ic_95']:>12.4f} {ctrl_result['std']**2:>12.4f}")

print(f"\nCoefficient β optimal : {ctrl_result['beta_opt']:.4f}")
print(f"Corrélation (Payoff, ST) : {ctrl_result['correlation']:.4f}")

reduction = 1 - (ctrl_result['std']**2) / (std_result['std']**2)
print(f"\nRéduction de variance : {reduction:.1%}")

## Exercice 6.3 : Calcul du Delta par Monte Carlo

**Énoncé** : Calculez le Delta d'un Call par différences finies et par la méthode pathwise.

In [ ]:
def delta_differences_finies(S0, K, T, r, sigma, N, epsilon=0.01, seed=42):
    """Delta par différences finies centrées."""
    np.random.seed(seed)
    Z = np.random.standard_normal(N)
    
    # Prix avec S0 + epsilon
    ST_up = (S0 + epsilon) * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    prix_up = np.exp(-r * T) * np.maximum(ST_up - K, 0).mean()
    
    # Prix avec S0 - epsilon
    ST_down = (S0 - epsilon) * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    prix_down = np.exp(-r * T) * np.maximum(ST_down - K, 0).mean()
    
    delta = (prix_up - prix_down) / (2 * epsilon)
    return delta


def delta_pathwise(S0, K, T, r, sigma, N, seed=42):
    """
    Delta par la méthode pathwise.
    
    On dérive le payoff : d/dS0 max(ST - K, 0) = 1{ST > K} × ST/S0
    """
    np.random.seed(seed)
    Z = np.random.standard_normal(N)
    
    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    
    # Dérivée du payoff
    dpayoff_dS0 = np.where(ST > K, ST / S0, 0)
    
    delta = np.exp(-r * T) * dpayoff_dS0.mean()
    std = np.exp(-r * T) * dpayoff_dS0.std(ddof=1) / np.sqrt(N)
    
    return delta, std


# Delta Black-Scholes
d1 = (np.log(S0/K) + (r + sigma**2/2) * T) / (sigma * np.sqrt(T))
delta_bs = norm.cdf(d1)

# Monte Carlo
N = 100000
delta_df = delta_differences_finies(S0, K, T, r, sigma, N)
delta_pw, delta_pw_std = delta_pathwise(S0, K, T, r, sigma, N)

print("CALCUL DU DELTA PAR MONTE CARLO\n")
print(f"{'Méthode':<30} {'Delta':>10} {'Erreur':>10}")
print("-" * 52)
print(f"{'Black-Scholes (référence)':<30} {delta_bs:>10.6f} {'-':>10}")
print(f"{'MC Différences finies':<30} {delta_df:>10.6f} {abs(delta_df - delta_bs):>10.6f}")
print(f"{'MC Pathwise':<30} {delta_pw:>10.6f} {abs(delta_pw - delta_bs):>10.6f}")
print(f"\nÉcart-type pathwise : {delta_pw_std:.6f}")

---

# Synthèse et Classe Générique de Valorisation

Pour conclure ce TD, voici une classe générique permettant de valoriser tout type d'option européenne ou exotique.

In [ ]:
class PricerUniversel:
    """
    Pricer universel pour options européennes et exotiques.
    
    Supporte :
    - Options vanille (call/put)
    - Options asiatiques
    - Options barrières
    - Options lookback
    - Options digitales
    
    Méthodes disponibles :
    - Black-Scholes (vanille uniquement)
    - Arbre binomial (CRR)
    - Monte Carlo (tous types)
    """
    
    def __init__(self, S0: float, K: float, T: float, r: float, sigma: float):
        self.S0 = S0
        self.K = K
        self.T = T
        self.r = r
        self.sigma = sigma
    
    def black_scholes(self, option_type: str = 'call') -> dict:
        """Prix et Greeks par Black-Scholes."""
        d1 = (np.log(self.S0/self.K) + (self.r + self.sigma**2/2) * self.T) / (self.sigma * np.sqrt(self.T))
        d2 = d1 - self.sigma * np.sqrt(self.T)
        
        if option_type == 'call':
            prix = self.S0 * norm.cdf(d1) - self.K * np.exp(-self.r * self.T) * norm.cdf(d2)
            delta = norm.cdf(d1)
        else:
            prix = self.K * np.exp(-self.r * self.T) * norm.cdf(-d2) - self.S0 * norm.cdf(-d1)
            delta = norm.cdf(d1) - 1
        
        gamma = norm.pdf(d1) / (self.S0 * self.sigma * np.sqrt(self.T))
        vega = self.S0 * norm.pdf(d1) * np.sqrt(self.T)
        theta = (-(self.S0 * norm.pdf(d1) * self.sigma) / (2 * np.sqrt(self.T)) 
                 - self.r * self.K * np.exp(-self.r * self.T) * norm.cdf(d2 if option_type == 'call' else -d2))
        
        return {
            'prix': prix,
            'delta': delta,
            'gamma': gamma,
            'vega': vega,
            'theta': theta
        }
    
    def monte_carlo(self, payoff_func, N: int = 100000, M: int = 252, 
                    antithetique: bool = True, seed: int = None) -> dict:
        """
        Valorisation Monte Carlo avec payoff personnalisable.
        
        Parameters
        ----------
        payoff_func : callable
            Fonction payoff(S, K) où S est une trajectoire (array 1D ou 2D)
        """
        if seed is not None:
            np.random.seed(seed)
        
        dt = self.T / M
        N_sim = N // 2 if antithetique else N
        
        # Génération des trajectoires
        Z = np.random.standard_normal((N_sim, M))
        
        X = np.zeros((N_sim, M + 1))
        X[:, 0] = np.log(self.S0)
        
        for i in range(M):
            X[:, i+1] = X[:, i] + (self.r - 0.5 * self.sigma**2) * dt + self.sigma * np.sqrt(dt) * Z[:, i]
        
        S = np.exp(X)
        payoffs = payoff_func(S, self.K)
        
        if antithetique:
            # Trajectoires antithétiques
            X_anti = np.zeros((N_sim, M + 1))
            X_anti[:, 0] = np.log(self.S0)
            
            for i in range(M):
                X_anti[:, i+1] = X_anti[:, i] + (self.r - 0.5 * self.sigma**2) * dt - self.sigma * np.sqrt(dt) * Z[:, i]
            
            S_anti = np.exp(X_anti)
            payoffs_anti = payoff_func(S_anti, self.K)
            payoffs = (payoffs + payoffs_anti) / 2
        
        prix = np.exp(-self.r * self.T) * payoffs.mean()
        std = np.exp(-self.r * self.T) * payoffs.std(ddof=1)
        ic_95 = 1.96 * std / np.sqrt(len(payoffs))
        
        return {'prix': prix, 'std': std, 'ic_95': ic_95}


# Fonctions de payoff prédéfinies
def payoff_call(S, K):
    return np.maximum(S[:, -1] - K, 0)

def payoff_put(S, K):
    return np.maximum(K - S[:, -1], 0)

def payoff_asian_call(S, K):
    return np.maximum(S.mean(axis=1) - K, 0)

def payoff_lookback_call(S, K):
    return S[:, -1] - S.min(axis=1)

def payoff_digital_call(S, K, payout=1):
    return payout * (S[:, -1] > K).astype(float)

def payoff_barrier_down_out_call(S, K, barrier=80):
    knocked = S.min(axis=1) <= barrier
    return np.where(knocked, 0, np.maximum(S[:, -1] - K, 0))


# Démonstration
pricer = PricerUniversel(S0=100, K=100, T=1, r=0.05, sigma=0.20)

print("PRICER UNIVERSEL - DÉMONSTRATION\n")
print(f"Paramètres : S₀=100, K=100, T=1, r=5%, σ=20%\n")
print(f"{'='*70}")

# Black-Scholes
bs = pricer.black_scholes('call')
print(f"\n1. BLACK-SCHOLES (Call vanille)")
print(f"   Prix : {bs['prix']:.4f}")
print(f"   Delta: {bs['delta']:.4f} | Gamma: {bs['gamma']:.4f} | Vega: {bs['vega']:.4f}")

# Monte Carlo - différents payoffs
print(f"\n2. MONTE CARLO (N=100,000, antithétique)")

mc_call = pricer.monte_carlo(payoff_call, N=100000, seed=42)
print(f"   Call vanille     : {mc_call['prix']:.4f} ± {mc_call['ic_95']:.4f}")

mc_asian = pricer.monte_carlo(payoff_asian_call, N=100000, seed=42)
print(f"   Call asiatique   : {mc_asian['prix']:.4f} ± {mc_asian['ic_95']:.4f}")

mc_lookback = pricer.monte_carlo(payoff_lookback_call, N=100000, seed=42)
print(f"   Call lookback    : {mc_lookback['prix']:.4f} ± {mc_lookback['ic_95']:.4f}")

mc_barrier = pricer.monte_carlo(
    lambda S, K: payoff_barrier_down_out_call(S, K, barrier=80), N=100000, seed=42
)
print(f"   Call barrière 80 : {mc_barrier['prix']:.4f} ± {mc_barrier['ic_95']:.4f}")

mc_digital = pricer.monte_carlo(
    lambda S, K: payoff_digital_call(S, K, payout=10), N=100000, seed=42
)
print(f"   Digital (10€)    : {mc_digital['prix']:.4f} ± {mc_digital['ic_95']:.4f}")

print(f"\n{'='*70}")

---

## Conclusion

Ce TD vous a permis de parcourir les concepts fondamentaux de valorisation des produits financiers :

1. **Obligations et zéro-coupons** : base de l'actualisation et stripping de courbe
2. **AOA et change à terme** : principe fondamental de la finance de marché
3. **Modèle binomial** : pricing par réplication et probabilité risque-neutre
4. **Arbre CRR** : extension multi-périodes et convergence vers Black-Scholes
5. **Monte Carlo** : méthode universelle pour options exotiques
6. **Réduction de variance** : techniques d'optimisation numérique

**Points clés à retenir :**
- L'AOA est le fondement de toute valorisation de produit dérivé
- La probabilité risque-neutre $\mathbb{Q}$ permet de valoriser par espérance actualisée
- Monte Carlo est incontournable dès que le produit dépend de la trajectoire
- Les techniques de réduction de variance sont essentielles en pratique

---

*TD réalisé dans le cadre du cours de Comptabilité Prudentielle - Master Actuariat*